<div style="background:#102A43;color:#ffffff;border-radius:11px;padding:30px 28px 24px;border-top:7px solid #18B7AC;">
<div style="color:#67DED3;letter-spacing:2px;font-weight:700;font-size:12px;">INTRODUCTION TO MACHINE LEARNING · GAZA SKY GEEKS</div>
<div style="font-size:31px;font-weight:750;line-height:1.25;margin:12px 0 9px;">Lecture 10 · Feature Selection, PCA,<br>and Reliable Preprocessing Pipelines</div>
<div style="font-size:15px;color:#DDEAF3;">Instructor: Diaeddin Rimawi &nbsp; | &nbsp; 17 November 2026 &nbsp; | &nbsp; Part 2: Data Preprocessing and Proximity Measures</div>
<div style="font-size:17px;color:#ffffff;border-top:1px solid #466178;padding-top:16px;margin-top:18px;"><strong>Today's challenge:</strong> Can we describe our penguins with fewer features, understand the information we preserve or lose, and apply every preprocessing decision reliably?</div>
</div>

We will continue with **Palmer Penguins** and the candidate representation created in L08. **No predictive model will be trained in this lecture.**

### What you will be able to do

By the end of L10, you should be able to:

- Explain why **more features** do not automatically mean **more useful information**.
- Inspect feature redundancy and apply **simple filtering and feature-ranking methods**.
- Distinguish **feature selection** (retain original columns) from **feature extraction** (form new coordinates).
- Explain PCA using **variance, new axes, and projection**, then interpret explained variance.
- Recognize what PCA might **lose**, including some nearest-neighbor relationships.
- Use **`fit`**, **`transform`**, **`Pipeline`**, and **`ColumnTransformer`** to process training and held-out observations consistently.
- Check the resulting numerical representation for **missing values, shape, alignment, and leakage**.

**Central principle:** A good representation is not just a small table. Its features, lost information, and transformation rules must all be justified.

### The question L09 left us

In L09, we saw that the closest penguin to a chosen penguin changed when we changed the **features**, even while using the same distance formula.

- Four standardized physical measurements emphasized physical similarity.
- Adding one-hot **island** indicators changed what counted as close.
- Adding **bill-shape ratio** could give related bill measurements extra influence.

> **If the features we choose can change our conclusions, how can we decide which features to keep, which to combine, and how to apply those choices consistently?**

We will answer that question in one investigation: **question the features → select or reduce → inspect what changed → make the process reliable**.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">01 · LECTURE 10</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Return to L09: does every feature deserve a place?</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Changing a representation can change the nearest observation.</div>
</div>

### Reopen the same scientific observations

- The local Palmer Penguins CSV contains **344 rows**; two have missing physical measurements.
- Our main teaching table contains **342 complete-measurement rows**, while preserving their original CSV row indices.
- `species` remains the **target**, never a distance feature or PCA input.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

NAVY, TEAL, GOLD, BLUE = "#102A43", "#18B7AC", "#DB9137", "#557AB5"
SPECIES_COLORS = {"Adelie": "#158B94", "Chinstrap": GOLD, "Gentoo": BLUE}
MEASUREMENTS = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
NUMERIC = MEASUREMENTS + ["bill_shape_ratio"]
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold"})

csv_path = Path("data/penguins.csv")
if not csv_path.is_file():
    raise FileNotFoundError("Place the existing course file at data/penguins.csv, next to this notebook.")
penguins = pd.read_csv(csv_path)
assert penguins.shape == (344, 8), "Check the course penguins.csv file."
study = penguins.dropna(subset=MEASUREMENTS).copy()
assert len(study) == 342
study["bill_shape_ratio"] = study["bill_length_mm"] / study["bill_depth_mm"]

# Same illustrative indices and split construction as L08/L09 (not student splits).
rng = np.random.default_rng(8)
order = rng.permutation(study.index.to_numpy())
n_train, n_val = int(.70 * len(study)), int(.20 * len(study))
study["split_role"] = "test"
study.loc[order[:n_train], "split_role"] = "train"
study.loc[order[n_train:n_train+n_val], "split_role"] = "validation"
train_mask = study["split_role"].eq("train")
y_species = study["species"].copy()  # interpretation/ranking only; NOT a PCA or distance input
print("Rows in working table:", len(study))
print("Illustrative roles:", study["split_role"].value_counts().to_dict())
display(study.loc[[0, 6, 145], MEASUREMENTS + ["island", "species"]])

### One quick reminder: feature choice changes proximity

Using **training statistics** for standardization, compare row 0 to all other penguins:

- **Representation A:** four standardized physical measurements.
- **Representation B:** the same measurements **plus** one-hot island indicators.

We already know Euclidean distance. Lets notice that **the input coordinates define what the formula measures**.

In [ ]:
# Compact recall of the final L09 comparison; all statistics/categories come from training.
train_means = study.loc[train_mask, MEASUREMENTS].mean()
train_sds = study.loc[train_mask, MEASUREMENTS].std(ddof=1)
physical_z = (study[MEASUREMENTS] - train_means) / train_sds
training_islands = sorted(study.loc[train_mask, "island"].dropna().unique())
island_encoded = pd.get_dummies(
    pd.Series(pd.Categorical(study["island"], categories=training_islands), index=study.index),
    prefix="island", dtype=int)
with_island = pd.concat([physical_z, island_encoded], axis=1)

def nearest_to(reference_idx, table, candidates=None, k=5):
    """Euclidean distances from an existing row to the other candidate rows."""
    choices = table if candidates is None else table.loc[candidates]
    values = choices.to_numpy(dtype=float)
    ref = table.loc[reference_idx].to_numpy(dtype=float)
    d = pd.Series(np.linalg.norm(values - ref, axis=1), index=choices.index)
    return d.drop(index=reference_idx, errors="ignore").sort_values(kind="stable").head(k)

physical_near = nearest_to(0, physical_z)
island_near = nearest_to(0, with_island)
print("Closest using four physical measurements:", physical_near.index[0])
print("Closest after including island:", island_near.index[0])
display(pd.DataFrame({"physical_only": physical_near.index.tolist(),
                      "physical_plus_island": island_near.index.tolist()},
                     index=pd.Index(range(1, 6), name="neighbor_rank")))

**Interpretation:** In this illustrative comparison, the closest observation changes from **row 145** to **row 6**. We have not established which is biologically or predictively “better”; we changed the meaning of similarity.

**Next:** Before calculating another distance, should we question the usefulness of our input columns?

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">02 · LECTURE 10</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Useful, redundant, or irrelevant?</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">More columns can add information—or repeat or distract from it.</div>
</div>

### Three reasons to question a feature

- **Constant or almost constant:** The feature scarcely distinguishes observations. If every row had the same `year`, it could not help separate them.
- **Redundant or related:** `body_mass_g` and `flipper_length_mm` are correlated; `bill_shape_ratio` is mathematically derived from two existing measurements.
- **Irrelevant for the question:** A feature may contain real variation but have little bearing on the comparison or prediction we intend to make.

These are **different problems**. A feature can be variable yet irrelevant, correlated yet useful, or derived yet valuable.

<!-- **Important distinction:** Correlation *between two inputs* is not the same as association *between one input and `species`*. -->

In [ ]:
# One figure: the same training observations viewed as a scatter and correlation matrix.
train_numeric = study.loc[train_mask, NUMERIC].copy()
corr = train_numeric.corr()
fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.0), constrained_layout=True)
axes[0].scatter(train_numeric["flipper_length_mm"], train_numeric["body_mass_g"],
                color=TEAL, alpha=.64, s=24)
axes[0].set(xlabel="Flipper length (mm)", ylabel="Body mass (g)",
            title="Related measurements in training rows")
axes[0].grid(alpha=.16)
heat = axes[1].imshow(corr.to_numpy(), cmap="BrBG", vmin=-1, vmax=1)
axes[1].set_xticks(range(len(NUMERIC)), ["Bill length", "Bill depth", "Flipper", "Mass", "Bill ratio"],
                   rotation=40, ha="right", fontsize=9)
axes[1].set_yticks(range(len(NUMERIC)), ["Bill length", "Bill depth", "Flipper", "Mass", "Bill ratio"], fontsize=9)
axes[1].set_title("Pearson correlation (training rows)")
fig.colorbar(heat, ax=axes[1], shrink=.72, label="Pearson r")
plt.show()
print("Flipper length / body mass correlation in training:",
      round(corr.loc["flipper_length_mm", "body_mass_g"], 3))

### What does this evidence tell us?

- Pearson's **$r$** describes the direction and strength of a *linear* relationship (from $-1$ to $1$).
- A large $|r|$ suggests two measurements **overlap in the variation they describe**; it does not prove that one can be removed without consequence.
- The ratio uses existing measurements, but can make a useful proportion **easier for a future algorithm to express**.
- All observed correlations depend on the data subset. A pattern across all species need not hold with the same strength inside each species.

**Next:** How can we make simple, explicit feature-selection decisions?

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">03 · LECTURE 10</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Practical feature selection: three simple filters</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Keep original columns, but use evidence for each decision.</div>
</div>

### Selection method 1 — Remove features without variation

A **variance filter** uses the spread of each input column:

$$\operatorname{Var}(X_j)=\frac{1}{n}\sum_{i=1}^{n}(x_{ij}-\bar x_j)^2.$$

- If every training observation has the same value, variance is **0**: the feature cannot distinguish training rows.
- `VarianceThreshold(threshold=0)` keeps columns with **nonzero training variance**.
- The rule is **unsupervised**: it does not inspect `species`.
- Small variance does **not** automatically mean small importance, particularly when measurement units differ.

We will temporarily add an invented constant column to our existing penguin table to illustrate the filter; it is not a real penguin attribute.

In [ ]:
with_constant = train_numeric.assign(constant_example=1)
nonconstant_filter = VarianceThreshold(threshold=0)
nonconstant_filter.fit(with_constant)  # fit on training inputs only
print("Input columns:", with_constant.columns.tolist())
print("Columns retained:", with_constant.columns[nonconstant_filter.get_support()].tolist())
print("Columns removed:", with_constant.columns[~nonconstant_filter.get_support()].tolist())

### Selection method 2 — Inspect correlated inputs

A **correlation-based filter** might flag a pair when $|r|$ exceeds a chosen threshold (for example, **0.85**).

- This is a **screening rule**, not a mathematical guarantee that one feature is useless.
- The threshold is a **choice**; changing it may change the result.
- After finding a pair, deciding which column to retain requires considering **meaning, measurement quality, availability, and purpose**.
- We compute the correlation from **training inputs only**; we must not inspect test-set results to make a feature decision.

For this small dataset we can inspect the strongest pair directly, without building a complicated automatic pruning algorithm.

In [ ]:
upper = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
strongest_pair = upper.stack().sort_values(ascending=False).head(3)
print("Three strongest absolute correlations among training numeric features:")
display(strongest_pair.rename("absolute_Pearson_r").round(3).to_frame())
print("Pairs flagged at |r| >= 0.85:", int((upper >= .85).sum().sum()))
# We do NOT delete columns automatically: statistical overlap is not proof of irrelevance.

### Selection method 3 — Rank features using the training labels

What if our *future* goal is to identify `species`? We may legitimately use the **training species labels** to evaluate which individual measurements vary among species.

A simple **ANOVA F-score** compares two types of variation:

$$F\ \approx\ \frac{\text{variation between species-group means}}{\text{variation within species groups}}.$$

- **Larger F:** stronger evidence that the numerical feature has different group means in these training observations.
- **Smaller F:** group means look more similar relative to within-group spread.
- This scores **one feature at a time**. It does not capture every nonlinear relationship or feature interaction.
- It is a **filter**, **not** a classifier and **not** measured predictive accuracy.
- We may use `y_train` here, but **never validation/test labels** to calculate the ranking.

`SelectKBest` implements ranking and selecting the top $k$ original columns. We use $k=3$ only to show its behavior, not as an established optimal number.

In [ ]:
ranker = SelectKBest(score_func=f_classif, k=3)
ranker.fit(train_numeric, y_species.loc[train_numeric.index])  # training inputs + training labels ONLY
ranking = pd.DataFrame({"feature": NUMERIC, "F_score": ranker.scores_,
                        "selected_top3": ranker.get_support()}).sort_values("F_score", ascending=False)
display(ranking.round(2))
print("Selected original columns:", ranker.get_feature_names_out().tolist())

fig, ax = plt.subplots(figsize=(9.2, 3.4))
ordered = ranking.iloc[::-1]
ax.barh(ordered["feature"], ordered["F_score"],
        color=[TEAL if selected else "#B9C8D3" for selected in ordered["selected_top3"]])
ax.set(xlabel="ANOVA F-score (training data only)",
       title="A supervised filter ranks individual features")
ax.grid(axis="x", alpha=.15)
plt.show()

### What did the three methods actually decide?

| Method | Looks at | Answers | Does **not** answer |
|:--|:--|:--|:--|
| Variance filter | Training inputs | Does this column vary? | Is it useful for `species`? |
| Correlation screen | Pairs of training inputs | Do two columns vary together? | Can we safely discard one? |
| Univariate F-ranking | Training inputs **and** training `species` | Which individual columns distinguish group means? | Which feature subset predicts best? |

**Terminology:** These are **filter methods**. *Wrapper* and *embedded* selection use predictive models to make selection decisions; we will return to such methods after learning model training.

**Next:** Could we reduce dimensions without having to choose which original measurements to discard?

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">04 · LECTURE 10</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Feature selection versus feature extraction</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Keep some existing measurements—or create new coordinates.</div>
</div>

### Two different responses to too many features

| **Feature selection** | **Feature extraction** |
|:--|:--|
| Retain some original columns | Construct new columns from the originals |
| e.g., keep `flipper_length_mm` and `body_mass_g` | e.g., create `PC1` and `PC2` as combinations of measurements |
| Original units and meanings remain | New directions have less direct physical meaning |
| Removed columns may contain lost information | Discarded directions may contain lost information |

Think of each observation as a point with **five numerical coordinates**. PCA will find a different coordinate system, arranged so that earlier axes describe more of the data's variation.

**Crucial:** PCA does **not** choose the three “best” original measurements. It creates **three new numerical components**.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">05 · LECTURE 10</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Discover PCA through geometry</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Can a diagonal direction summarize two related measurements?</div>
</div>

### Why rotate the axes?

Imagine two features that tend to increase together. A 2D scatter plot looks like a narrow, slanted cloud.

1. The cloud has a **center** (the mean of each feature).
2. Its largest spread often lies along a **slanted direction**, rather than the original horizontal or vertical axes.
3. PCA finds the direction that captures the **greatest variance**: **principal component 1 (PC1)**.
4. The next direction, **PC2**, is **perpendicular** and captures as much remaining variance as possible.
5. If we keep only PC1, we turn a 2D observation into **one coordinate**, but lose its position along PC2.

First, use a tiny controlled 2D cloud to understand the geometry; then we will return to the real penguins.

In [ ]:
# Controlled geometry example: correlated coordinates (not a second real dataset).
toy_rng = np.random.default_rng(12)
toy_x = np.linspace(-3.5, 3.5, 40)
toy_points = np.column_stack([toy_x, 0.65 * toy_x + toy_rng.normal(0, .4, len(toy_x))])
toy_pca = PCA(n_components=2).fit(toy_points)
center = toy_pca.mean_
axis1, axis2 = toy_pca.components_
strengths = np.sqrt(toy_pca.explained_variance_)

fig, ax = plt.subplots(figsize=(9.1, 4.6))
ax.scatter(toy_points[:, 0], toy_points[:, 1], color=BLUE, alpha=.74, s=28, label="Original points")
ax.scatter(*center, color=NAVY, s=72, marker="x", linewidths=2.5, label="Center (mean)")
for direction, length, color, label in [
    (axis1, strengths[0] * 2.0, TEAL, "PC1: most variation"),
    (axis2, strengths[1] * 2.6, GOLD, "PC2: perpendicular")]:
    endpoints = np.vstack([center-length*direction, center+length*direction])
    ax.plot(endpoints[:, 0], endpoints[:, 1], color=color, lw=3, label=label)
ax.set(xlabel="Original feature 1", ylabel="Original feature 2",
       title="PCA finds new directions of variation")
ax.set_aspect("equal", adjustable="box")
ax.legend(frameon=False, loc="upper left", fontsize=9)
ax.grid(alpha=.14)
plt.show()
print("PC1 explained variance:", f"{toy_pca.explained_variance_ratio_[0]:.1%}")
print("PC2 explained variance:", f"{toy_pca.explained_variance_ratio_[1]:.1%}")

### What is a projection?

PCA first **centers** each point by subtracting the training mean. A component is a **unit direction** $\mathbf{u}_1$; a point's new coordinate is its shadow on this axis:

$$z_1=(\mathbf{x}-\boldsymbol{\mu}_{\mathrm{train}})\cdot\mathbf{u}_1.$$

For a simple worked calculation (the direction is provided, not learned by hand):

- Observation: $\mathbf{x}=(3,2)$; center: $\boldsymbol{\mu}=(1,1)$.
- Centered point: $(2,1)$; unit direction: $\mathbf{u}_1=(0.8,0.6)$.
- Projection: $z_1=2(0.8)+1(0.6)=\mathbf{2.2}$.

The value **2.2 is not a millimeter or gram measurement**: it is a coordinate along a new direction.

For multiple components, projection gives a new vector $(z_1,z_2,\ldots)$. The directions are mutually perpendicular (**orthogonal**) in the feature space used to fit PCA.

In [ ]:
x = np.array([3., 2.]); mu = np.array([1., 1.]); u1 = np.array([.8, .6])
print("Centered point:", x - mu)
print("Coordinate along the new axis:", np.dot(x - mu, u1))
print("Unit length of the direction:", np.linalg.norm(u1))

### Reducing from two dimensions to one

If we keep **both** components, the data are rotated into a new coordinate system (no dimensionality reduction). If we keep only **PC1**:

- Each original point is stored as **one number**.
- We can approximately reconstruct a point on the PC1 line, but not recover its discarded PC2 coordinate.
- The difference between the original point and this reconstruction is **information lost by the projection**.
- PCA chooses the direction that minimizes total squared reconstruction error for one-component linear projection—equivalently, the direction that retains the greatest variance.

Do not confuse **high retained variance** with **high prediction quality**. PCA never looked at a prediction target.

In [ ]:
toy_one = PCA(n_components=1).fit(toy_points)
toy_1d = toy_one.transform(toy_points)
toy_reconstructed = toy_one.inverse_transform(toy_1d)
fig, ax = plt.subplots(figsize=(8.8, 4.3))
for p, rec in zip(toy_points[::3], toy_reconstructed[::3]):
    ax.plot([p[0], rec[0]], [p[1], rec[1]], color="#9CACB8", lw=1.2, alpha=.8)
ax.scatter(toy_points[:, 0], toy_points[:, 1], color=BLUE, s=32, alpha=.6, label="Original (2D)")
ax.scatter(toy_reconstructed[:, 0], toy_reconstructed[:, 1], color=TEAL, s=24,
           alpha=.85, label="Approximate reconstruction (1D)")
ax.set(xlabel="Original feature 1", ylabel="Original feature 2",
       title="Keeping PC1: each point is projected onto a line")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=.14); ax.legend(frameon=False)
plt.show()
print("Original shape:", toy_points.shape, "→ projected shape:", toy_1d.shape)
print("Reconstructed shape:", toy_reconstructed.shape)

**PCA in one sentence:** Find new orthogonal directions ordered by retained variance; keep fewer of them when an approximate representation is acceptable.

**A key limitation to test:** Does preserving almost all *total* variance guarantee preserving *every pairwise distance or nearest neighbor*?

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">06 · LECTURE 10</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Apply PCA to our five numerical penguin features</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">How much measured variation can fewer components retain?</div>
</div>

### Return to the engineered representation from L08

Our five numerical candidate features are the four physical measurements and the derived **bill-shape ratio**. They use different scales, so we first standardize them using **training** values.

- **Scaling before PCA matters:** otherwise the larger numerical units can dominate the direction of maximum variance.
- `PCA` **centers** its input internally but does **not** standardize each feature automatically.
- Fit both `StandardScaler` and PCA **only on training rows**, then apply their learned parameters to other rows.
- We will keep `island` separate: applying PCA only to continuous numerical features avoids mixing nominal category indicators into this first explanation.

The explained variance ratio for component $i$ is:

$$\text{EVR}_i=\frac{\text{variance along PC}_i}{\text{sum of variances along all PCs}}.$$

A common **illustrative** rule is to keep enough PCs to reach **95% cumulative explained variance**. The threshold is a choice, not proof that the result is best for every task.

In [ ]:
# These fit operations use training rows only. Index and y_species remain separate.
numeric_scaler = StandardScaler()
X_train_scaled = numeric_scaler.fit_transform(study.loc[train_mask, NUMERIC])
X_all_scaled = numeric_scaler.transform(study[NUMERIC])
full_pca = PCA().fit(X_train_scaled)
ratios = full_pca.explained_variance_ratio_
cumulative = np.cumsum(ratios)
n_keep = int(np.searchsorted(cumulative, .95) + 1)
reduced_pca = PCA(n_components=n_keep).fit(X_train_scaled)
all_components = pd.DataFrame(reduced_pca.transform(X_all_scaled),
                              index=study.index, columns=[f"PC{i+1}" for i in range(n_keep)])
print("Numerical input dimensions:", len(NUMERIC))
print("Components needed for at least 95% *training* variance:", n_keep)
print("Training variance retained:", f"{cumulative[n_keep-1]:.1%}")
display(pd.DataFrame({"component": [f"PC{i+1}" for i in range(len(ratios))],
                      "variance_ratio": ratios, "cumulative": cumulative}).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.0), constrained_layout=True)
for species, color in SPECIES_COLORS.items():
    rows = study.index[study["species"].eq(species)]
    axes[0].scatter(all_components.loc[rows, "PC1"], all_components.loc[rows, "PC2"],
                    s=23, alpha=.65, color=color, label=species)
axes[0].set(xlabel="Principal component 1", ylabel="Principal component 2",
            title="Penguins in a new numerical coordinate system")
axes[0].legend(frameon=False, fontsize=9); axes[0].grid(alpha=.15)
xs = np.arange(1, len(NUMERIC)+1)
axes[1].plot(xs, cumulative, marker="o", color=TEAL, lw=2.5)
axes[1].axhline(.95, color=GOLD, lw=1.5, ls="--", label="Illustrative 95% target")
axes[1].axvline(n_keep, color=NAVY, lw=1, ls=":")
axes[1].set(xlabel="Number of retained components", ylabel="Training variance retained",
            ylim=(0, 1.05), xticks=xs, title="Cumulative explained variance")
axes[1].legend(frameon=False, fontsize=8); axes[1].grid(alpha=.15)
plt.show()

### Read these PCA results carefully

- The components are **weighted combinations** of the numerical inputs, not selected original attributes.
- PC1 is the direction that explains the greatest **training variance**; PC2 explains the largest remaining variance while staying perpendicular to PC1.
- A two-component scatter plot is useful for **visualizing patterns**, even if we retain more components for the reduced feature table.
- Coloring the points by `species` helps us **interpret** the plot. **No species labels were used to fit PCA.**
- The **95%** criterion measures retained numerical variation. It does **not** mean 95% classification accuracy or 95% of every important relationship.

**L09 connection:** The distance between two points may change after we drop some principal components. Let's check proximity for one training reference.

In [ ]:
# Compare the same training observations in two numerical feature spaces.
train_ids = study.index[train_mask]
ref_id = int(train_ids[0])
original_five = pd.DataFrame(X_all_scaled, index=study.index, columns=NUMERIC)
near_full = nearest_to(ref_id, original_five, candidates=train_ids, k=5)
near_reduced = nearest_to(ref_id, all_components, candidates=train_ids, k=5)
comparison = pd.DataFrame({"five_standardized_features": near_full.index.tolist(),
                           f"{n_keep}_principal_components": near_reduced.index.tolist()},
                          index=pd.Index(range(1, 6), name="neighbor_rank"))
print("Training reference (original CSV row):", ref_id)
display(comparison)
print("Shared neighbors (ignoring their order):",
      len(set(near_full.index).intersection(near_reduced.index)), "/ 5")

**Interpretation:** This is evidence about *how proximity behaves under two representations*. We should compare **neighbor identities and rankings**, not claim that the smaller Euclidean distance value is inherently better across spaces with different dimensions.

**Next:** We understand the transformation. How do we reproduce it safely when training, validation, and test data are processed at different times?

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">07 · LECTURE 10</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Reusable transformations: fit, then transform</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">What should be learned once, and what should be reused?</div>
</div>

### Manual preprocessing becomes fragile as steps accumulate

In L07–L08 we manually handled missing values, created a ratio, calculated standardization statistics, and encoded island categories. The same operations must be applied to **future observations** without learning new parameters from them.

| Call | Meaning | Example |
|:--|:--|:--|
| `fit(X_train)` | **Learn** parameters from training data | median, mean/SD, island vocabulary, PCA directions |
| `transform(X_new)` | **Apply saved** parameters | transform validation/test or a new observation |
| `fit_transform(X_train)` | Fit and transform the **same training** data | convenient shorthand during training preparation |

**Rule:** Validation and test observations may be **transformed**, but they must not be used to **fit** preprocessing decisions. A pipeline does not enforce this rule for us if we call `fit` on the wrong subset.

### One detail when comparing with L08

L08 manually calculated standard deviations with Pandas `std(ddof=1)`. In scikit-learn, `StandardScaler` uses a population-style standard deviation (`ddof=0`).

- Both are standard conventions; small numerical differences are expected.
- Different numerical values **do not imply leakage**.
- When comparing the *baseline* and *PCA-reduced* representations below, we will use the **same scikit-learn scaling convention for both**, so that PCA is the substantive difference.

**Next:** Can we let a reusable object perform multiple transformations in the correct order?

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">08 · LECTURE 10</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Preprocessing pipelines for reliable feature tables</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Combine sequential steps and different column types—without fitting a model.</div>
</div>

### Step A — Sequential operations with `Pipeline`

A **pipeline** is an ordered sequence of fitted transformations. For our numerical columns:

**Numerical input → `SimpleImputer` → `StandardScaler` → numerical output**

- `SimpleImputer(strategy="median")` learns a median **from the training subset** for each numeric input.
- `StandardScaler` learns means and standard deviations of the **imputed training values**.
- `transform` runs the same steps, in the same order, for every later observation.

Our **342-row complete-measurement table** contains no missing entries in these columns, so the imputer does not change its values. We retain it to show the reusable workflow; it matters when later data have missing entries.

For this lecture the `bill_shape_ratio` was constructed *before* the pipeline with a fixed formula.

In [ ]:
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
X_train_raw = study.loc[train_mask, NUMERIC + ["island"]].copy()
X_val_raw = study.loc[study["split_role"].eq("validation"), NUMERIC + ["island"]].copy()
X_test_raw = study.loc[study["split_role"].eq("test"), NUMERIC + ["island"]].copy()

numeric_pipeline.fit(X_train_raw[NUMERIC])               # learn only from training
train_numeric_ready = numeric_pipeline.transform(X_train_raw[NUMERIC])
validation_numeric_ready = numeric_pipeline.transform(X_val_raw[NUMERIC])
print("Numerical training matrix:", train_numeric_ready.shape)
print("Numerical validation matrix:", validation_numeric_ready.shape)
print("Scaler learned means (first two):",
      np.round(numeric_pipeline.named_steps["scale"].mean_[:2], 2))

### Step B — Different column types need different operations

The five numerical features and the categorical `island` column **should not go through identical transformations**.

| Numerical branch | Island branch |
|:--|:--|
| Fill missing numeric values with training medians | Fill missing island values with training mode |
| Standardize using training statistics | One-hot encode using training vocabulary |
| Optionally apply PCA afterward | Keep the categorical indicators outside PCA |

`ColumnTransformer` applies each branch to its specified input columns, then **concatenates the results in a stable order**.

**Note:** `handle_unknown="ignore"` produces all-zero indicators for an unseen island. This avoids a runtime error but can be ambiguous; we still need a documented unknown-category policy.

In [ ]:
island_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
full_preparation = ColumnTransformer([
    ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUMERIC),
    ("island", island_pipeline, ["island"]),
])
full_pipeline = Pipeline([("prepare", full_preparation)])  # preprocessing ONLY; no classifier
full_pipeline.fit(X_train_raw)   # IMPORTANT: never fit on validation or test

def as_named_table(pipeline, inputs):
    arr = pipeline.transform(inputs)
    names = pipeline.named_steps["prepare"].get_feature_names_out()
    return pd.DataFrame(arr, index=inputs.index, columns=names)

baseline_train = as_named_table(full_pipeline, X_train_raw)
baseline_val = as_named_table(full_pipeline, X_val_raw)
baseline_test = as_named_table(full_pipeline, X_test_raw)
print("Prepared training/validation/test shapes:",
      baseline_train.shape, baseline_val.shape, baseline_test.shape)
print("Learned output names:", baseline_train.columns.tolist())
display(baseline_train.head(3).round(2))

### Step C — Put PCA **inside the numerical branch**

We do **not** need to build a new pipeline from scratch: we add PCA **after scaling** inside the numerical branch. Island encoding stays identical.

- **Baseline:** five standardized numerical columns + island indicators.
- **Reduced:** the selected principal components + the **same** island indicators.
- Both workflows see the same raw candidate inputs and use only **training** data to fit transformations.
- We use the component count chosen earlier from the **training** explained-variance curve.

This comparison isolates dimensionality reduction without silently changing categorical encoding or observation identities.

In [ ]:
reduced_numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=n_keep)),
])
reduced_preparation = ColumnTransformer([
    ("numeric", reduced_numeric_pipeline, NUMERIC),
    ("island", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore",
                                                   sparse_output=False))]), ["island"]),
])
reduced_pipeline = Pipeline([("prepare", reduced_preparation)])
reduced_pipeline.fit(X_train_raw)  # fit ALL learned steps using the same training subset
reduced_train = as_named_table(reduced_pipeline, X_train_raw)
reduced_val = as_named_table(reduced_pipeline, X_val_raw)
reduced_test = as_named_table(reduced_pipeline, X_test_raw)
print("Full representation:", baseline_train.shape[1], "columns")
print("PCA representation:", reduced_train.shape[1], "columns")
print("Reduced output names:", reduced_train.columns.tolist())
display(reduced_train.head(3).round(2))

### What the pipeline protects—and what it cannot protect

**A pipeline helps:**

- Apply operations in a reproducible **order**.
- Reuse the same fitted medians, scaling statistics, category vocabulary, and PCA axes.
- Produce consistent feature-column ordering across subsets.

**A pipeline does not automatically prevent:**

- Calling `fit` on the **whole dataset**, including held-out rows.
- Using test labels to **choose** features or PCA dimensions.
- Accidentally including `species` or `split_role` among predictive inputs.
- Creating the bill ratio differently across splits, or losing row identity before labels are joined.

For the course's **future** predictive tasks, validation and test partitions retain their own roles; today we only verify that we can transform them safely.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">9 · LECTURE 10</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Audit, summarize, and prepare for learning</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Is our final representation numerically valid and scientifically defensible?</div>
</div>

### A five-part check before any Machine Learning algorithm

1. **Rows:** Does each feature row still match the correct original observation and target?
2. **Roles:** Are `species` and `split_role` kept **outside** the input matrix?
3. **Values:** Are transformed inputs numerical and finite (no missing values or infinities)?
4. **Columns:** Did fit/transform produce the same feature names/order across train, validation, and test?
5. **Interpretation:** How many dimensions were removed, and what evidence do we have about lost variation and changed proximity?

Check these properties rather than trusting a visually plausible table.

In [ ]:
for label, raw, baseline, reduced in [
    ("train", X_train_raw, baseline_train, reduced_train),
    ("validation", X_val_raw, baseline_val, reduced_val),
    ("test", X_test_raw, baseline_test, reduced_test),
]:
    assert raw.index.equals(baseline.index) and raw.index.equals(reduced.index)
    assert np.isfinite(baseline.to_numpy()).all()
    assert np.isfinite(reduced.to_numpy()).all()
    assert baseline.columns.equals(baseline_train.columns)
    assert reduced.columns.equals(reduced_train.columns)
    assert "species" not in raw.columns and "split_role" not in raw.columns
    assert study.loc[raw.index, "species"].index.equals(raw.index)
    print(f"{label:10s}: {len(raw):3d} rows | original {len(NUMERIC)} numeric + island"
          f" | baseline {baseline.shape[1]} cols | PCA {reduced.shape[1]} cols | checks passed")
print("All checks passed. No predictive model has been fitted.")

### Beyond Tabular Data: What About Images?

So far, we have prepared **tabular data**, where each observation contains measurements such as bill length or body mass. But Machine Learning can also learn from images, audio, and text.

Images require their own preprocessing techniques. For example:

- **Resizing:** Adjust images to consistent dimensions, such as 128 × 128 pixels.
- **Normalization:** Scale pixel intensities, for example from 0–255 to 0–1.
- **Flattening:** Convert a 2D image into a 1D feature vector. A 28 × 28 grayscale image becomes **784 numerical features**. The pixel values are preserved, but the spatial arrangement is no longer explicit.
- **Noise reduction:** Reduce unwanted random variations that may affect image quality.
- **Gaussian filtering:** Smooth an image using a weighted average of neighboring pixels, giving greater importance to nearby pixels. This can reduce noise but also blur important edges and details.

**The connection to our lecture:**

- Different preprocessing decisions change what information a Machine Learning algorithm receives.
- Images can produce very high-dimensional feature representations.
- Techniques such as PCA can sometimes reduce those dimensions while preserving important variation.

**Remember:** There is no universally best preprocessing method. The appropriate transformations depend on the data and the task. Modern image-learning methods may also preserve spatial structure instead of flattening images.

### Our investigation, in five ideas

| Question | What we learned |
|:--|:--|
| Do we need every feature? | Question constant, overlapping, and task-irrelevant information. |
| How can we keep fewer originals? | Use domain knowledge or simple **filter methods**; training labels are legitimate for supervised filters. |
| Can we create fewer new coordinates? | **PCA** orders orthogonal directions by retained variance. |
| What might reduction change? | Even high variance retention may change **nearest neighbors** and hide local differences. |
| How do we apply the same decisions safely? | Fit preprocessing **on training only**, transform held-out observations, and **audit** results. |

**Part 2 is complete:** We can now prepare and check numeric inputs responsibly, ready for algorithms that learn from them.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">10 · LECTURE 10</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Assignment 05</div>
<div style="color:#DDEAF3;font-size:14px;margin-top:5px;">Investigate one justified improvement to your own feature representation.</div>
</div>

## Assignment 05 — Building a Reliable Preprocessing Pipeline

### The task

In **Assignment 04**, you prepared the Palmer Penguins dataset manually. Now, your task is to **automate the same preprocessing operations using scikit-learn pipelines**, apply PCA to reduce dimensionality, and save the processed dataset.

### Instructions

1. **Load your dataset**
   - Continue using the CSV file from Assignment 04, keeping the original input columns available for preprocessing.
   - Preserve the existing `train`, `val`, and `test` assignments. Do not generate new splits.
   - Separate `species` (target) and `split` (metadata) from the input features.

2. **Build a preprocessing pipeline** using `Pipeline` and `ColumnTransformer`:
   - **Missing numerical values:** Use `SimpleImputer` with the same strategy you chose in Assignment 04 (`mean` or `median`).
   - **Missing categorical values:** Use `SimpleImputer` with `most_frequent` (mode).
   - **Feature engineering:** Include the `bill_shape_ratio` feature from Assignment 04.
   - **Numerical scaling:** Use `StandardScaler`.
   - **Categorical encoding:** Use `OneHotEncoder` for `island`.
   - **Dimensionality reduction:** Apply PCA to the numerical features only, retaining at least **95% of their training variance**. Keep the encoded categorical features.

3. **Apply the pipeline**
   - Use `fit_transform()` on the training observations only.
   - Use `transform()` on validation and test observations without refitting.
   - Ensure all observations receive consistent transformations.

4. **Save the processed dataset**
   - Combine the processed features from all three splits.
   - Restore the original observation identities, `species`, and `split` columns.
   - Preserve the original row order and split assignments.
   - Save the result as **`penguins_processed.csv`**.

5. **Verify your result**
   - Confirm that the processed features are numerical and contain no missing values.
   - Display the original and processed feature counts.
   - Report the number of PCA components and their cumulative explained variance.
   - Confirm that all original observations and split assignments are preserved.

### Important rules

- All preprocessing parameters must be learned from **training data only**.
- Do not use `species` or `split` as input features.
- Do not standardize already-standardized columns again.
- The pipeline should reproduce the preprocessing decisions from Assignment 04 while adding PCA.
- Do not train a Machine Learning model.

### Submission

Submit the following **three files**:

1. **`Assignment05.ipynb`** — Your complete, runnable code that loads the original CSV, applies the preprocessing pipeline, and generates the processed CSV.
2. **Your original CSV file** — The input dataset used by your notebook.
3. **`penguins_processed.csv`** — The final dataset generated by your pipeline.

**Goal:** Build a reproducible preprocessing workflow that automatically transforms your original data into a clean, numerical, lower-dimensional dataset ready for Machine Learning.

### Looking ahead — L11: Linear Regression

Across L07–L10 we learned how to **clean**, **engineer**, **compare**, **select/reduce**, and **reliably transform** features.

But none of these steps by itself teaches an algorithm how to predict a target.

> **If our observations are now well represented, how can a Machine Learning algorithm learn a numerical relationship from training examples and make predictions for unseen observations?**

That is where **L11 — Linear Regression: Predicting Continuous Values** begins.

### Sources, acknowledgments, and further reading

- **Palmer Penguins:** Horst, A. M., Hill, A. P., & Gorman, K. B. (2020), *palmerpenguins: Palmer Archipelago (Antarctica) penguin data* ([project](https://allisonhorst.github.io/palmerpenguins/); [dataset DOI](https://doi.org/10.5281/zenodo.3960218)). Field observations described by Gorman, Williams & Fraser (2014). We use the course's **existing local CSV**.
- **Teaching inspiration:** Prof. Antonio Liotta's shared preprocessing material and introductory PCA lecture informed the topic progression. All explanatory figures and examples here are newly constructed for L10.
- **scikit-learn:** [Feature selection](https://scikit-learn.org/stable/modules/feature_selection.html), [`SelectKBest`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html), [PCA](https://scikit-learn.org/stable/modules/decomposition.html#pca), [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [`ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), and [common pitfalls / data leakage](https://scikit-learn.org/stable/common_pitfalls.html).

**Offline execution:** Requires Python, NumPy, Pandas, Matplotlib, scikit-learn, IPython/Jupyter, and the course's existing `data/penguins.csv`. No downloads, API keys, or model training are needed. The lecture uses an illustrative split only; students must retain their own original Assignment 03 splits.